# Full ImageBind-style Scratch Training on ESC-50

This notebook is a course-scale from-scratch implementation inspired by ImageBind. It trains an audio-text shared embedding model from scratch on ESC-50.

- This is not a reproduction of Meta AI's web-scale ImageBind training.
- The model is initialized randomly and trained for 5 epochs in the main run.
- No pretrained ImageBind weights are loaded.
- No frozen ImageBind embeddings are used.
- The priority is correct from-scratch training for 5 epochs, checkpoint saving, and test evaluation for the course mini-implementation requirement.

## 1. Setup

Imports, reproducibility settings, device detection, and output folders.

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

try:
    from sklearn.metrics import confusion_matrix
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

try:
    import torchaudio
    HAS_TORCHAUDIO = True
except Exception as exc:
    HAS_TORCHAUDIO = False
    torchaudio = None
    print(f'torchaudio unavailable, using fallback audio utilities where needed: {exc}')

try:
    import soundfile as sf
    HAS_SOUNDFILE = True
except Exception:
    HAS_SOUNDFILE = False
    sf = None

try:
    from scipy.io import wavfile
    HAS_SCIPY_WAV = True
except Exception:
    HAS_SCIPY_WAV = False
    wavfile = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

CHECKPOINT_DIR = Path('checkpoints')
OUTPUT_DIR = Path('outputs')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. ESC-50 Dataset Loading

The dataset root is easy to edit. If the default Kaggle path is not present, the notebook searches common Kaggle input folders for `meta/esc50.csv`.

Fold split:
- Train: folds 1, 2, 3
- Validation: fold 4
- Test: fold 5

In [ ]:
# Edit this path if your Kaggle ESC-50 dataset is mounted somewhere else.
ESC50_ROOT = '/kaggle/input/esc50-dataset/ESC-50-master'

SAMPLE_RATE = 16000
CLIP_DURATION = 5.0
NUM_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION)
N_FFT = 1024
HOP_LENGTH = 320
N_MELS = 128
F_MIN = 50.0
F_MAX = 8000.0

TRAIN_FOLDS = [1, 2, 3]
VAL_FOLDS = [4]
TEST_FOLDS = [5]


def find_esc50_root(preferred_root: str) -> Path:
    preferred = Path(preferred_root)
    if (preferred / 'meta' / 'esc50.csv').exists():
        return preferred

    search_roots = [Path('/kaggle/input'), Path.cwd()]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        try:
            candidates.extend(root.rglob('meta/esc50.csv'))
        except Exception as exc:
            print(f'Skipping search under {root}: {exc}')

    for metadata_path in candidates:
        esc_root = metadata_path.parent.parent
        if (esc_root / 'audio').exists():
            return esc_root

    searched = ', '.join(str(p) for p in search_roots)
    raise FileNotFoundError(
        f'Could not find ESC-50. Set ESC50_ROOT to the folder containing meta/esc50.csv. Searched: {searched}'
    )

ESC50_ROOT = find_esc50_root(ESC50_ROOT)
META_PATH = ESC50_ROOT / 'meta' / 'esc50.csv'
AUDIO_DIR = ESC50_ROOT / 'audio'

print(f'ESC-50 root: {ESC50_ROOT}')
print(f'Metadata: {META_PATH}')
print(f'Audio dir: {AUDIO_DIR}')

metadata = pd.read_csv(META_PATH)
metadata['fold'] = metadata['fold'].astype(int)
metadata['category'] = metadata['category'].astype(str)

classes = sorted(metadata['category'].unique().tolist())
class2idx = {name: idx for idx, name in enumerate(classes)}
idx2class = {idx: name for name, idx in class2idx.items()}
metadata['label'] = metadata['category'].map(class2idx).astype(int)

train_df = metadata[metadata['fold'].isin(TRAIN_FOLDS)].reset_index(drop=True)
val_df = metadata[metadata['fold'].isin(VAL_FOLDS)].reset_index(drop=True)
test_df = metadata[metadata['fold'].isin(TEST_FOLDS)].reset_index(drop=True)

print(f'Classes: {len(classes)}')
print(f'Train samples: {len(train_df)} | Val samples: {len(val_df)} | Test samples: {len(test_df)}')
display(train_df[['filename', 'fold', 'category', 'label']].head())

## 3. Audio Preprocessing and Dataset

Each clip is converted to mono, resampled to 16 kHz, padded or cropped to 5 seconds, and converted into a normalized log-mel spectrogram.

In [ ]:
def load_audio_mono(path: Path) -> Tuple[torch.Tensor, int]:
    if HAS_TORCHAUDIO:
        waveform, sr = torchaudio.load(str(path))
        waveform = waveform.float()
        if waveform.ndim == 2:
            waveform = waveform.mean(dim=0)
        return waveform, int(sr)

    if HAS_SOUNDFILE:
        data, sr = sf.read(str(path), dtype='float32', always_2d=False)
        if data.ndim == 2:
            data = data.mean(axis=1)
        return torch.from_numpy(np.asarray(data, dtype=np.float32)), int(sr)

    if HAS_SCIPY_WAV:
        sr, data = wavfile.read(str(path))
        if data.ndim == 2:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        max_abs = np.max(np.abs(data)) if data.size else 1.0
        if max_abs > 0:
            data = data / max_abs
        return torch.from_numpy(data), int(sr)

    raise ImportError('Need torchaudio, soundfile, or scipy to load ESC-50 wav files.')


def resample_if_needed(waveform: torch.Tensor, orig_sr: int, target_sr: int) -> torch.Tensor:
    if orig_sr == target_sr:
        return waveform
    if HAS_TORCHAUDIO:
        return torchaudio.functional.resample(waveform, orig_freq=orig_sr, new_freq=target_sr)

    old_len = waveform.numel()
    new_len = int(round(old_len * target_sr / orig_sr))
    waveform_3d = waveform.view(1, 1, old_len)
    resampled = F.interpolate(waveform_3d, size=new_len, mode='linear', align_corners=False)
    return resampled.view(-1)


def pad_or_crop(waveform: torch.Tensor, num_samples: int) -> torch.Tensor:
    if waveform.numel() < num_samples:
        return F.pad(waveform, (0, num_samples - waveform.numel()))
    return waveform[:num_samples]


class LogMelExtractor(nn.Module):
    def __init__(self, sample_rate: int, n_fft: int, hop_length: int, n_mels: int, f_min: float, f_max: float):
        super().__init__()
        if HAS_TORCHAUDIO:
            self.mel = torchaudio.transforms.MelSpectrogram(
                sample_rate=sample_rate,
                n_fft=n_fft,
                hop_length=hop_length,
                n_mels=n_mels,
                f_min=f_min,
                f_max=f_max,
                power=2.0,
            )
        else:
            self.mel = None
            self.n_fft = n_fft
            self.hop_length = hop_length
            self.n_mels = n_mels
            self.register_buffer('window', torch.hann_window(n_fft), persistent=False)
            freq_bins = n_fft // 2 + 1
            edges = torch.linspace(0, freq_bins, n_mels + 1).long()
            filters = torch.zeros(n_mels, freq_bins)
            for i in range(n_mels):
                start, end = int(edges[i]), int(edges[i + 1])
                end = max(end, start + 1)
                filters[i, start:end] = 1.0 / (end - start)
            self.register_buffer('filters', filters, persistent=False)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        if self.mel is not None:
            mel = self.mel(waveform)
        else:
            spec = torch.stft(
                waveform,
                n_fft=self.n_fft,
                hop_length=self.hop_length,
                window=self.window.to(waveform.device),
                return_complex=True,
            ).abs().pow(2.0)
            mel = self.filters.to(waveform.device) @ spec
        log_mel = torch.log(mel.clamp_min(1e-6))
        log_mel = (log_mel - log_mel.mean()) / (log_mel.std().clamp_min(1e-6))
        return log_mel


mel_extractor = LogMelExtractor(SAMPLE_RATE, N_FFT, HOP_LENGTH, N_MELS, F_MIN, F_MAX)


class ESC50LogMelDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, audio_dir: Path, mel_extractor: nn.Module):
        self.df = dataframe.reset_index(drop=True).copy()
        self.audio_dir = Path(audio_dir)
        self.mel_extractor = mel_extractor

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        waveform, sr = load_audio_mono(self.audio_dir / row['filename'])
        waveform = resample_if_needed(waveform, sr, SAMPLE_RATE)
        waveform = pad_or_crop(waveform, NUM_SAMPLES)
        with torch.no_grad():
            mel = self.mel_extractor(waveform).unsqueeze(0)
        label = int(row['label'])
        return mel.float(), torch.tensor(label, dtype=torch.long)


train_dataset = ESC50LogMelDataset(train_df, AUDIO_DIR, mel_extractor)
val_dataset = ESC50LogMelDataset(val_df, AUDIO_DIR, mel_extractor)
test_dataset = ESC50LogMelDataset(test_df, AUDIO_DIR, mel_extractor)

sample_mel, sample_label = train_dataset[0]
print(f'Sample mel shape: {tuple(sample_mel.shape)} | label: {sample_label.item()} ({idx2class[sample_label.item()]})')

## 4. Text Prompts / Class Text Inputs

Each class receives a prompt of the form `a sound of {class_name}`. The text/class encoder is a randomly initialized learnable embedding table trained from scratch. It does not use CLIP, sentence transformers, or any pretrained text encoder.

In [ ]:
class_prompts = [f'a sound of {name.replace("_", " ")}' for name in classes]
for i, prompt in enumerate(class_prompts[:10]):
    print(f'{i:02d}: {prompt}')

## 5. Full ImageBind-style Scratch Model

The model maps audio and text/class inputs into a shared L2-normalized embedding space. It includes an AST-style audio encoder, randomly initialized class text embeddings, audio and text projection heads, and a learned temperature scale.

In [ ]:
class AudioPatchTransformerEncoder(nn.Module):
    def __init__(
        self,
        patch_size: Tuple[int, int] = (16, 16),
        embed_dim: int = 256,
        depth: int = 4,
        num_heads: int = 4,
        mlp_ratio: float = 4.0,
        dropout: float = 0.1,
        max_tokens: int = 512,
    ):
        super().__init__()
        self.patch_embed = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, max_tokens, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, mel: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(mel)
        x = x.flatten(2).transpose(1, 2)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        if x.size(1) > self.pos_embed.size(1):
            raise ValueError(f'Too many audio tokens: {x.size(1)} > {self.pos_embed.size(1)}')
        x = x + self.pos_embed[:, :x.size(1)]
        x = self.transformer(x)
        return self.norm(x[:, 0])


class FullScratchImageBindAudioText(nn.Module):
    def __init__(self, num_classes: int, audio_width: int = 256, text_width: int = 256, shared_embed_dim: int = 256, temperature: float = 0.07):
        super().__init__()
        self.audio_encoder = AudioPatchTransformerEncoder(embed_dim=audio_width)
        self.class_text_embeddings = nn.Embedding(num_classes, text_width)
        self.audio_projection = nn.Sequential(nn.LayerNorm(audio_width), nn.Linear(audio_width, shared_embed_dim))
        self.text_projection = nn.Sequential(nn.LayerNorm(text_width), nn.Linear(text_width, shared_embed_dim))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / temperature)))
        self._init_text_and_projection()

    def _init_text_and_projection(self):
        nn.init.normal_(self.class_text_embeddings.weight, std=0.02)
        for module in [self.audio_projection[-1], self.text_projection[-1]]:
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode_audio(self, mel: torch.Tensor) -> torch.Tensor:
        audio_features = self.audio_encoder(mel)
        audio_embeddings = self.audio_projection(audio_features)
        return F.normalize(audio_embeddings, dim=-1)

    def encode_text(self, class_ids: torch.Tensor) -> torch.Tensor:
        text_features = self.class_text_embeddings(class_ids)
        text_embeddings = self.text_projection(text_features)
        return F.normalize(text_embeddings, dim=-1)

    def forward(self, mel: torch.Tensor, class_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.encode_audio(mel), self.encode_text(class_ids)


model = FullScratchImageBindAudioText(num_classes=len(classes), shared_embed_dim=256, temperature=0.07).to(device)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.__class__.__name__)
print(f'Parameters: {num_params:,} | Trainable: {trainable_params:,}')

## 6. Symmetric Contrastive Loss / InfoNCE

The batch similarity matrix compares audio embeddings with matching class text embeddings. Diagonal entries are positive audio-text pairs. The objective averages audio-to-text and text-to-audio cross entropy losses.

In [ ]:
def symmetric_contrastive_loss(audio_embeddings: torch.Tensor, text_embeddings: torch.Tensor, logit_scale: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    scale = logit_scale.exp().clamp(max=100.0)
    logits = scale * audio_embeddings @ text_embeddings.t()
    targets = torch.arange(logits.size(0), device=logits.device)
    loss_a2t = F.cross_entropy(logits, targets)
    loss_t2a = F.cross_entropy(logits.t(), targets)
    loss = 0.5 * (loss_a2t + loss_t2a)
    return loss, logits


def batch_top1_from_logits(logits: torch.Tensor) -> float:
    targets = torch.arange(logits.size(0), device=logits.device)
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

## 7. Training From Scratch

The default run trains for 5 epochs. Increase `NUM_EPOCHS` only if extra runtime is available. The epoch-1 checkpoint is saved to `checkpoints/full_scratch_epoch1.pth` and training history is saved to `outputs/scratch_full_train_history.json`.

In [ ]:
NUM_EPOCHS = 5
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2 if os.name != 'nt' else 0
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(NUM_EPOCHS, 1))


def run_contrastive_epoch(model, loader, optimizer=None) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for mel, labels in loader:
            mel = mel.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            audio_embeddings, text_embeddings = model(mel, labels)
            loss, logits = symmetric_contrastive_loss(audio_embeddings, text_embeddings, model.logit_scale)

            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            total_acc += batch_top1_from_logits(logits)
            total_batches += 1

    return {'loss': total_loss / max(total_batches, 1), 'top1': total_acc / max(total_batches, 1)}


history = []
for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_contrastive_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_contrastive_epoch(model, val_loader, optimizer=None)
    scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'],
        'train_top1': train_metrics['top1'],
        'val_loss': val_metrics['loss'],
        'val_top1': val_metrics['top1'],
        'lr': optimizer.param_groups[0]['lr'],
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={row['train_loss']:.4f} train_top1={row['train_top1']:.4f} | "
        f"val_loss={row['val_loss']:.4f} val_top1={row['val_top1']:.4f}"
    )

    if epoch == 1:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'class2idx': class2idx,
            'idx2class': idx2class,
            'class_prompts': class_prompts,
            'config': {
                'sample_rate': SAMPLE_RATE,
                'clip_duration': CLIP_DURATION,
                'n_fft': N_FFT,
                'hop_length': HOP_LENGTH,
                'n_mels': N_MELS,
                'shared_embed_dim': 256,
                'temperature': 0.07,
                'train_folds': TRAIN_FOLDS,
                'val_folds': VAL_FOLDS,
                'test_folds': TEST_FOLDS,
            },
        }
        torch.save(checkpoint, CHECKPOINT_DIR / 'full_scratch_epoch1.pth')
        print(f"Saved checkpoint: {CHECKPOINT_DIR / 'full_scratch_epoch1.pth'}")

with open(OUTPUT_DIR / 'scratch_full_train_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f"Saved training history: {OUTPUT_DIR / 'scratch_full_train_history.json'}")

## 8. Evaluation on ESC-50 Test Split

Classification compares each test audio embedding against all class text embeddings. Retrieval metrics are computed in both directions over the test split.

In [ ]:
@torch.no_grad()
def compute_class_text_embeddings(model, num_classes: int) -> torch.Tensor:
    model.eval()
    class_ids = torch.arange(num_classes, device=device, dtype=torch.long)
    return model.encode_text(class_ids)


@torch.no_grad()
def evaluate_classification(model, loader, num_classes: int) -> Dict[str, object]:
    model.eval()
    class_text_embeddings = compute_class_text_embeddings(model, num_classes)
    total = 0
    top1_correct = 0
    top5_correct = 0
    all_labels = []
    all_preds = []
    all_audio_embeddings = []

    for mel, labels in loader:
        mel = mel.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        audio_embeddings = model.encode_audio(mel)
        logits = audio_embeddings @ class_text_embeddings.t()
        top1 = logits.argmax(dim=1)
        top5 = logits.topk(k=min(5, num_classes), dim=1).indices

        total += labels.numel()
        top1_correct += (top1 == labels).sum().item()
        top5_correct += (top5 == labels.unsqueeze(1)).any(dim=1).sum().item()
        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(top1.cpu().tolist())
        all_audio_embeddings.append(audio_embeddings.cpu())

    return {
        'test_top1': top1_correct / max(total, 1),
        'test_top5': top5_correct / max(total, 1),
        'labels': all_labels,
        'preds': all_preds,
        'audio_embeddings': torch.cat(all_audio_embeddings, dim=0),
        'labels_tensor': torch.tensor(all_labels, dtype=torch.long),
    }


@torch.no_grad()
def compute_retrieval_metrics(model, eval_outputs: Dict[str, object], recall_ks=(1, 5, 10)) -> Dict[str, float]:
    model.eval()
    audio_embeddings = eval_outputs['audio_embeddings'].to(device)
    labels = eval_outputs['labels_tensor'].to(device)
    text_embeddings = compute_class_text_embeddings(model, len(classes))
    recalls = {}

    a2t_scores = audio_embeddings @ text_embeddings.t()
    a2t_ranked = a2t_scores.argsort(dim=1, descending=True)
    for k in recall_ks:
        k_eff = min(k, text_embeddings.size(0))
        hit = (a2t_ranked[:, :k_eff] == labels.unsqueeze(1)).any(dim=1).float().mean().item()
        recalls[f'audio_to_text_R@{k}'] = hit

    t2a_scores = text_embeddings @ audio_embeddings.t()
    t2a_ranked = t2a_scores.argsort(dim=1, descending=True)
    for k in recall_ks:
        k_eff = min(k, audio_embeddings.size(0))
        hits = []
        for class_id in range(text_embeddings.size(0)):
            retrieved_labels = labels[t2a_ranked[class_id, :k_eff]]
            hits.append((retrieved_labels == class_id).any().float())
        recalls[f'text_to_audio_R@{k}'] = torch.stack(hits).mean().item()

    return recalls


eval_outputs = evaluate_classification(model, test_loader, len(classes))
retrieval_recalls = compute_retrieval_metrics(model, eval_outputs)
labels = eval_outputs['labels']
preds = eval_outputs['preds']

per_class_accuracy = {}
for class_id, class_name in idx2class.items():
    idxs = [i for i, y in enumerate(labels) if y == class_id]
    per_class_accuracy[class_name] = float(np.mean([preds[i] == labels[i] for i in idxs])) if idxs else None

results = {
    'model': 'Full ImageBind-style scratch audio-text model',
    'train_from_scratch': True,
    'pretrained_weights': False,
    'dataset': 'ESC-50',
    'test_fold': TEST_FOLDS,
    'epochs': NUM_EPOCHS,
    'test_top1': eval_outputs['test_top1'],
    'test_top5': eval_outputs['test_top5'],
    'per_class_accuracy': per_class_accuracy,
    'checkpoint': str(CHECKPOINT_DIR / 'full_scratch_epoch1.pth'),
}

with open(OUTPUT_DIR / 'scratch_full_eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
with open(OUTPUT_DIR / 'scratch_full_retrieval_recall.json', 'w') as f:
    json.dump(retrieval_recalls, f, indent=2)

print(f"Test Top-1: {results['test_top1']:.4f}")
print(f"Test Top-5: {results['test_top5']:.4f}")
print('Retrieval recalls:')
for key, value in retrieval_recalls.items():
    print(f'  {key}: {value:.4f}')
print(f"Saved evaluation results: {OUTPUT_DIR / 'scratch_full_eval_results.json'}")
print(f"Saved retrieval recalls: {OUTPUT_DIR / 'scratch_full_retrieval_recall.json'}")

## 9. Plots

The notebook saves the learning curve and confusion matrix into `outputs/`.

In [ ]:
def save_learning_curve(history: List[Dict[str, float]], output_path: Path):
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    val_loss = [row['val_loss'] for row in history]
    train_top1 = [row['train_top1'] for row in history]
    val_top1 = [row['val_top1'] for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, marker='o', label='train loss')
    axes[0].plot(epochs, val_loss, marker='o', label='val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Contrastive Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, train_top1, marker='o', label='train top-1')
    axes[1].plot(epochs, val_top1, marker='o', label='val top-1')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Batch retrieval top-1')
    axes[1].set_title('Training/Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches='tight')
    plt.show()


def save_confusion_matrix(labels: List[int], preds: List[int], output_path: Path):
    if HAS_SKLEARN:
        cm = confusion_matrix(labels, preds, labels=list(range(len(classes))))
    else:
        cm = np.zeros((len(classes), len(classes)), dtype=np.int64)
        for true_label, pred_label in zip(labels, preds):
            cm[true_label, pred_label] += 1

    fig_size = max(12, len(classes) * 0.28)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    class_names = [idx2class[i] for i in range(len(classes))]
    if HAS_SEABORN:
        sns.heatmap(cm, ax=ax, cmap='Blues', square=True, cbar=True, xticklabels=class_names, yticklabels=class_names)
    else:
        im = ax.imshow(cm, cmap='Blues')
        fig.colorbar(im, ax=ax)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(class_names, rotation=90)
        ax.set_yticklabels(class_names)

    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title('ESC-50 Test Confusion Matrix')
    plt.xticks(rotation=90, fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()


save_learning_curve(history, OUTPUT_DIR / 'scratch_full_learning_curve.png')
save_confusion_matrix(labels, preds, OUTPUT_DIR / 'scratch_full_confusion_matrix.png')
print(f"Saved learning curve: {OUTPUT_DIR / 'scratch_full_learning_curve.png'}")
print(f"Saved confusion matrix: {OUTPUT_DIR / 'scratch_full_confusion_matrix.png'}")

## 10. Summary Table

This table records the required scratch run and points to the saved checkpoint.

In [ ]:
summary_table = pd.DataFrame([
    {
        'Model': 'Full ImageBind-style scratch audio-text model',
        'Train from scratch': 'Yes',
        'Pretrained weights': 'No',
        'Dataset': 'ESC-50',
        'Epochs': NUM_EPOCHS,
        'Test Top-1': f"{results['test_top1']:.4f}",
        'Test Top-5': f"{results['test_top5']:.4f}",
        'Checkpoint': str(CHECKPOINT_DIR / 'full_scratch_epoch1.pth'),
    }
])
summary_table

## Final Note

This scratch experiment satisfies the implementation requirement because the model is initialized from random weights and trained directly on ESC-50 for 5 epochs. The result is not expected to match pretrained ImageBind because the original model uses web-scale image-text/audio-video data and large pretrained encoders.

## ADDON: Rubric completion and missing components

This addon preserves all original cells and only standardizes missing rubric deliverables at the end of the notebook.

It writes new files to /kaggle/working/imagebind_rubric_outputs/01_esc50_audio/ on Kaggle, with a local ./kaggle_working fallback. Optional visualizations and ablations are wrapped in try/except; if the needed runtime variables are unavailable, the addon writes NOT_RUN or FAILED_WITH_REASON instead of inventing results.

ESC-50 addon adds standardized metadata, final evaluation summary, temperature sweep, prompt-template benchmark status, recall bar, and rubric report. It explicitly records that fold 5 only was evaluated.

In [ ]:
from pathlib import Path
import json, math, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
ADDON_DIR = ROOT_OUT / "imagebind_rubric_outputs" / "01_esc50_audio"
ADDON_DIR.mkdir(parents=True, exist_ok=True)
FAST_ABLATION = True

def _jsonable(x):
    if isinstance(x, dict): return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [_jsonable(v) for v in x]
    if hasattr(x, "tolist"): return x.tolist()
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    return x

def _load_json_candidates(names):
    roots = [ROOT_OUT, Path.cwd(), Path("ResultForSound1")]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                with open(p, "r", encoding="utf-8") as f:
                    return json.load(f), str(p)
    return None, None

eval_json, eval_source = _load_json_candidates(["scratch_full_eval_results.json"])
retrieval_json, retrieval_source = _load_json_candidates(["scratch_full_retrieval_recall.json"])
history_json, history_source = _load_json_candidates(["scratch_full_train_history.json"])

runtime_results = globals().get("results", None)
runtime_retrieval = globals().get("retrieval_recalls", None)
runtime_history = globals().get("history", None)

eval_summary = {
    "dataset": "ESC-50",
    "modality": "Audio-Text",
    "model_name": "FullScratchImageBindAudioText",
    "train_from_scratch": True,
    "pretrained_weights": False,
    "epochs": globals().get("NUM_EPOCHS", eval_json.get("epochs") if isinstance(eval_json, dict) else "UNKNOWN"),
    "loss": "symmetric InfoNCE / contrastive loss",
    "main_metrics": {},
    "ablation_status": "ADDED_BY_RUBRIC_ADDON",
    "limitation": "Only fold 5 is evaluated in this notebook; the full ESC-50 protocol is a 5-fold average.",
    "sources": {"eval": eval_source, "retrieval": retrieval_source, "history": history_source},
}
src_eval = runtime_results if isinstance(runtime_results, dict) else (eval_json or {})
src_ret = runtime_retrieval if isinstance(runtime_retrieval, dict) else (retrieval_json or {})
for k in ["test_top1", "test_top5"]:
    if k in src_eval: eval_summary["main_metrics"][k] = float(src_eval[k])
for k, v in src_ret.items():
    if "R@" in k: eval_summary["main_metrics"][k] = float(v)

with open(ADDON_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(eval_summary), f, indent=2)

# Temperature sweep. For ranking metrics, temperature is rank-invariant when applied as a scalar to logits.
temperature_values = [0.05, 0.07, 0.1, 0.2]
temperature_ablation = []
try:
    import torch
    model_obj = globals().get("model", None)
    eval_outputs_obj = globals().get("eval_outputs", None)
    classes_obj = globals().get("classes", None)
    device_obj = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if model_obj is None or eval_outputs_obj is None or classes_obj is None:
        raise RuntimeError("Runtime model/eval_outputs/classes are unavailable; rerun original training/evaluation cells first.")
    model_obj.eval()
    audio_emb = eval_outputs_obj["audio_embeddings"].to(device_obj)
    labels_tensor = eval_outputs_obj["labels_tensor"].to(device_obj)
    text_emb = compute_class_text_embeddings(model_obj, len(classes_obj)).to(device_obj)
    base_scores = audio_emb @ text_emb.t()
    for tau in temperature_values:
        logits = base_scores / tau
        loss = torch.nn.functional.cross_entropy(logits, labels_tensor).item()
        top1 = (logits.argmax(dim=1) == labels_tensor).float().mean().item()
        top5 = (logits.topk(k=min(5, logits.size(1)), dim=1).indices == labels_tensor.unsqueeze(1)).any(dim=1).float().mean().item()
        row = {"temperature": tau, "classification_ce": loss, "test_top1": top1, "test_top5": top5, "note": "Scalar temperature changes CE but not ranking order."}
        for kk, vv in src_ret.items():
            if "R@" in kk: row[kk] = float(vv)
        temperature_ablation.append(row)
except Exception as exc:
    temperature_ablation = [{"status": "NOT_RUN", "reason": str(exc), "temperatures": temperature_values}]

with open(ADDON_DIR / "esc50_ablation_temperature.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(temperature_ablation), f, indent=2)
with open(ADDON_DIR / "ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(temperature_ablation), f, indent=2)

# Prompt template benchmark: this scratch model uses one learned class embedding per class, not a tokenized text encoder.
templates = ["a sound of {}", "this is a sound of {}", "an audio recording of {}", "{} sound", "the sound of {}"]
prompt_ablation = []
try:
    if "class_prompts" in globals() and "classes" in globals():
        for template in templates:
            prompt_ablation.append({
                "template": template,
                "status": "NOT_RUN",
                "reason": "Model text side is a learned class embedding table; prompt strings are not tokenized, so changing text templates does not change embeddings.",
            })
    else:
        raise RuntimeError("classes/class_prompts unavailable.")
except Exception as exc:
    prompt_ablation = [{"status": "NOT_RUN", "reason": str(exc), "templates": templates}]
with open(ADDON_DIR / "esc50_prompt_template_ablation.json", "w", encoding="utf-8") as f:
    json.dump(_jsonable(prompt_ablation), f, indent=2)

try:
    ret = eval_summary["main_metrics"]
    labels_bar = ["A2T R@1", "A2T R@5", "A2T R@10", "T2A R@1", "T2A R@5", "T2A R@10"]
    keys = ["audio_to_text_R@1", "audio_to_text_R@5", "audio_to_text_R@10", "text_to_audio_R@1", "text_to_audio_R@5", "text_to_audio_R@10"]
    vals = [ret.get(k, np.nan) for k in keys]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(labels_bar, vals)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Recall")
    ax.set_title("ESC-50 Audio-Text Retrieval Recall")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(ADDON_DIR / "esc50_recall_bar.png", dpi=160)
    fig.savefig(ADDON_DIR / "retrieval_bar.png", dpi=160)
    plt.show()
except Exception as exc:
    print("Recall bar skipped:", exc)

try:
    hist = runtime_history if isinstance(runtime_history, list) else history_json
    if hist:
        df = pd.DataFrame(hist)
        fig, ax = plt.subplots(figsize=(7, 4))
        if "train_loss" in df: ax.plot(df["epoch"], df["train_loss"], marker="o", label="train_loss")
        if "val_loss" in df: ax.plot(df["epoch"], df["val_loss"], marker="o", label="val_loss")
        ax.set_title("ESC-50 Learning Curve")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
        fig.tight_layout(); fig.savefig(ADDON_DIR / "learning_curve.png", dpi=160); plt.show()
except Exception as exc:
    print("Learning curve addon skipped:", exc)

report = f"""ESC-50 rubric addon report
===========================
Overview: audio-text ImageBind-style scratch training on ESC-50.
Method: audio log-mel input is encoded and aligned with learned class text embeddings using symmetric InfoNCE. ImageBind's core idea is a shared embedding space; this mini version uses audio-text pairs/classes instead of web-scale pretrained encoders.
Implementation: train_from_scratch=True, pretrained_weights=False, epochs={eval_summary['epochs']}.
Evaluation: {json.dumps(eval_summary['main_metrics'], indent=2)}
Ablation: temperature sweep saved to esc50_ablation_temperature.json. Prompt template benchmark is marked NOT_RUN because the text side is a learned class embedding table, not a tokenized prompt encoder.
Limitations: Only fold 5 was evaluated; full ESC-50 protocol is 5-fold average. Small scratch model can overfit.
Future work: 5-fold evaluation, stronger augmentations, tokenized text encoder, longer training, and full multimodal contrastive alignment.
"""
with open(ADDON_DIR / "esc50_rubric_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
with open(ADDON_DIR / "01_esc50_audio_rubric_report.txt", "w", encoding="utf-8") as f:
    f.write(report)
print("ADDON outputs:", ADDON_DIR)
print(json.dumps(eval_summary, indent=2))

## ADDON: Export standardized outputs

This cell standardizes this notebook's outputs into its required project folder. It copies available files, creates a short README/report if needed, and never crashes when optional outputs are missing.

In [ ]:
from pathlib import Path
import json
import shutil
import traceback

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
STANDARD_OUTPUT_DIR = WORKING_DIR / "ResultForSound1"
STANDARD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_STEM = "01_esc50_audio"
DATASET_NAME = "ESC-50"
MODALITY_NAME = "Audio-Text"
MAX_CHECKPOINT_BYTES = 100 * 1024 * 1024
CHECKPOINT_SUFFIXES = {".pth", ".pt", ".ckpt"}

FILE_EXPORTS = {
    "train_history.json": ["train_history.json", "scratch_full_train_history.json"],
    "eval_results.json": ["eval_results.json", "scratch_full_eval_results.json"],
    "final_eval_summary.json": ["final_eval_summary.json"],
    "retrieval_recall.json": ["retrieval_recall.json", "scratch_full_retrieval_recall.json"],
    "ablation_summary.json": ["ablation_summary.json", "esc50_ablation_temperature.json"],
    "learning_curve.png": ["learning_curve.png", "scratch_full_learning_curve.png"],
    "confusion_matrix.png": ["confusion_matrix.png", "scratch_full_confusion_matrix.png"],
    "report.txt": ["report.txt", "esc50_5epoch_short_report.txt", "esc50_10epoch_short_report.txt", "esc50_rubric_report.txt"],
    "rubric_report.txt": ["rubric_report.txt", "esc50_rubric_report.txt", "01_esc50_audio_rubric_report.txt"],
    "esc50_ablation_temperature.json": ["esc50_ablation_temperature.json"],
    "esc50_prompt_template_ablation.json": ["esc50_prompt_template_ablation.json"],
    "esc50_recall_bar.png": ["esc50_recall_bar.png", "retrieval_bar.png"],
}

def _as_path(value):
    try:
        return Path(value)
    except Exception:
        return None

def _candidate_dirs():
    dirs = [
        WORKING_DIR,
        WORKING_DIR / "outputs",
        WORKING_DIR / "results",
        WORKING_DIR / "figures",
        WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM,
        Path("."),
        Path("outputs"),
        Path("results"),
        Path("figures"),
        Path("ResultForSound1"),
    ]
    for var_name in ["OUTPUT_DIR", "RESULT_DIR", "FIGURE_DIR"]:
        if var_name in globals():
            p = _as_path(globals()[var_name])
            if p is not None:
                dirs.append(p)
                dirs.append(p / "results")
                dirs.append(p / "figures")
    seen = set()
    out = []
    for d in dirs:
        try:
            key = str(d.resolve()) if d.exists() else str(d)
        except Exception:
            key = str(d)
        if key not in seen:
            out.append(d)
            seen.add(key)
    return out

def _skip_reason(path):
    try:
        if path.suffix.lower() in CHECKPOINT_SUFFIXES and path.stat().st_size > MAX_CHECKPOINT_BYTES:
            return f"large checkpoint >100MB ({path.stat().st_size / (1024**2):.1f} MB)"
    except Exception as exc:
        return f"stat failed: {exc}"
    return None

def _copy_file(src, dst):
    reason = _skip_reason(src)
    if reason:
        return {"source": str(src), "dest": str(dst), "status": "SKIPPED", "reason": reason}
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            if src.resolve() == dst.resolve():
                return {"source": str(src), "dest": str(dst), "status": "ALREADY_IN_PLACE"}
        except Exception:
            pass
        shutil.copy2(src, dst)
        return {"source": str(src), "dest": str(dst), "status": "COPIED"}
    except Exception as exc:
        return {"source": str(src), "dest": str(dst), "status": "FAILED", "reason": repr(exc)}

def _find_source(names):
    for base in _candidate_dirs():
        for name in names:
            p = Path(name)
            candidates = []
            if p.is_absolute():
                candidates.append(p)
            else:
                candidates.append(base / name)
                try:
                    if base.exists():
                        candidates.extend(base.rglob(name))
                except Exception:
                    pass
            for cand in candidates:
                if cand.exists() and cand.is_file():
                    return cand
    return None

manifest = []
for dest_name, source_names in FILE_EXPORTS.items():
    src = _find_source(source_names)
    dst = STANDARD_OUTPUT_DIR / dest_name
    if src is None:
        msg = {"dest": dest_name, "status": "MISSING", "searched_names": source_names}
        manifest.append(msg)
        print("WARNING missing output for", dest_name, "searched:", source_names)
        continue
    manifest.append(_copy_file(src, dst))

# Copy any extra rubric addon files into the standardized folder without overwriting canonical names.
addon_dir = WORKING_DIR / "imagebind_rubric_outputs" / NOTEBOOK_STEM
if addon_dir.exists():
    for src in sorted(addon_dir.rglob("*")):
        if not src.is_file():
            continue
        dst = STANDARD_OUTPUT_DIR / src.name
        if dst.exists():
            continue
        manifest.append(_copy_file(src, dst))

if not (STANDARD_OUTPUT_DIR / "final_eval_summary.json").exists() and not (STANDARD_OUTPUT_DIR / "eval_results.json").exists():
    fallback_summary = {
        "dataset": DATASET_NAME,
        "modality": MODALITY_NAME,
        "status": "NOT_RUN",
        "reason": "No eval_results.json or final_eval_summary.json was found by the export addon.",
    }
    with open(STANDARD_OUTPUT_DIR / "final_eval_summary.json", "w", encoding="utf-8") as f:
        json.dump(fallback_summary, f, indent=2)
    manifest.append({"dest": "final_eval_summary.json", "status": "CREATED_NOT_RUN"})

if not any((STANDARD_OUTPUT_DIR / name).exists() for name in ["report.txt", "rubric_report.txt"]):
    report = f"""Standardized output report
Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Notebook: {NOTEBOOK_STEM}
Status: exported available files. Missing files are listed in export_manifest.json.
"""
    with open(STANDARD_OUTPUT_DIR / "report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    manifest.append({"dest": "report.txt", "status": "CREATED_MINIMAL"})

readme = f"""# {NOTEBOOK_STEM} standardized outputs

Dataset: {DATASET_NAME}
Modality: {MODALITY_NAME}
Folder: {STANDARD_OUTPUT_DIR}

This folder is created by the export addon. Missing optional files are warnings, not fatal errors.
See export_manifest.json for copied, missing, and skipped files.
"""
with open(STANDARD_OUTPUT_DIR / "README_outputs.md", "w", encoding="utf-8") as f:
    f.write(readme)

with open(STANDARD_OUTPUT_DIR / "export_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Standardized output directory:", STANDARD_OUTPUT_DIR)
print("Exported files now present:")
for p in sorted(STANDARD_OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(STANDARD_OUTPUT_DIR))

## ADDON: Excel report

This cell creates a styled Excel workbook from JSON/TXT artifacts already exported into this notebook output folder. Missing files are recorded as NOT_AVAILABLE instead of stopping the notebook.

In [ ]:
from pathlib import Path
import json

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
EXCEL_OUTPUT_DIR = WORKING_DIR / "ResultForSound1"
EXCEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_REPORT_PATH = EXCEL_OUTPUT_DIR / "01_esc50_audio_excel_report.xlsx"

METRIC_SET = "esc50"

METRIC_SPECS = {
    "esc50": [
        ("test_top1", ["test_top1", "top1", "top1_acc", "test_accuracy"], "Test top-1 classification accuracy"),
        ("test_top5", ["test_top5", "top5", "top5_acc"], "Test top-5 classification accuracy"),
        ("audio_to_text_R@1", ["audio_to_text_R@1", "audio_to_text_r1", "a2t_R@1"], "Audio-to-text Recall@1"),
        ("audio_to_text_R@5", ["audio_to_text_R@5", "audio_to_text_r5", "a2t_R@5"], "Audio-to-text Recall@5"),
        ("audio_to_text_R@10", ["audio_to_text_R@10", "audio_to_text_r10", "a2t_R@10"], "Audio-to-text Recall@10"),
        ("text_to_audio_R@1", ["text_to_audio_R@1", "text_to_audio_r1", "t2a_R@1"], "Text-to-audio Recall@1"),
        ("text_to_audio_R@5", ["text_to_audio_R@5", "text_to_audio_r5", "t2a_R@5"], "Text-to-audio Recall@5"),
        ("text_to_audio_R@10", ["text_to_audio_R@10", "text_to_audio_r10", "t2a_R@10"], "Text-to-audio Recall@10"),
    ],
    "flickr8k": [
        ("image_to_text_R@1", ["image_to_text_R@1", "image_to_text_r1", "i2t_R@1", "I2T R@1"], "Image-to-text Recall@1"),
        ("image_to_text_R@5", ["image_to_text_R@5", "image_to_text_r5", "i2t_R@5", "I2T R@5"], "Image-to-text Recall@5"),
        ("image_to_text_R@10", ["image_to_text_R@10", "image_to_text_r10", "i2t_R@10", "I2T R@10"], "Image-to-text Recall@10"),
        ("text_to_image_R@1", ["text_to_image_R@1", "text_to_image_r1", "t2i_R@1", "T2I R@1"], "Text-to-image Recall@1"),
        ("text_to_image_R@5", ["text_to_image_R@5", "text_to_image_r5", "t2i_R@5", "T2I R@5"], "Text-to-image Recall@5"),
        ("text_to_image_R@10", ["text_to_image_R@10", "text_to_image_r10", "t2i_R@10", "T2I R@10"], "Text-to-image Recall@10"),
    ],
    "llvip": [
        ("image_to_thermal_R@1", ["image_to_thermal_R@1", "image_to_thermal_r1", "i2t_R@1"], "Visible image-to-thermal Recall@1"),
        ("image_to_thermal_R@5", ["image_to_thermal_R@5", "image_to_thermal_r5", "i2t_R@5"], "Visible image-to-thermal Recall@5"),
        ("image_to_thermal_R@10", ["image_to_thermal_R@10", "image_to_thermal_r10", "i2t_R@10"], "Visible image-to-thermal Recall@10"),
        ("thermal_to_image_R@1", ["thermal_to_image_R@1", "thermal_to_image_r1", "t2i_R@1"], "Thermal-to-visible image Recall@1"),
        ("thermal_to_image_R@5", ["thermal_to_image_R@5", "thermal_to_image_r5", "t2i_R@5"], "Thermal-to-visible image Recall@5"),
        ("thermal_to_image_R@10", ["thermal_to_image_R@10", "thermal_to_image_r10", "t2i_R@10"], "Thermal-to-visible image Recall@10"),
        ("accuracy", ["accuracy", "test_accuracy", "top1", "top1_acc"], "Optional classification/top-1 accuracy"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 accuracy"),
    ],
    "nyu": [
        ("image_to_depth_R@1", ["image_to_depth_R@1", "image_to_depth_r1", "i2d_r@1", "I2D R@1"], "RGB image-to-depth Recall@1"),
        ("image_to_depth_R@5", ["image_to_depth_R@5", "image_to_depth_r5", "i2d_r@5", "I2D R@5"], "RGB image-to-depth Recall@5"),
        ("image_to_depth_R@10", ["image_to_depth_R@10", "image_to_depth_r10", "i2d_r@10", "I2D R@10"], "RGB image-to-depth Recall@10"),
        ("depth_to_image_R@1", ["depth_to_image_R@1", "depth_to_image_r1", "d2i_r@1", "D2I R@1"], "Depth-to-RGB image Recall@1"),
        ("depth_to_image_R@5", ["depth_to_image_R@5", "depth_to_image_r5", "d2i_r@5", "D2I R@5"], "Depth-to-RGB image Recall@5"),
        ("depth_to_image_R@10", ["depth_to_image_R@10", "depth_to_image_r10", "d2i_r@10", "D2I R@10"], "Depth-to-RGB image Recall@10"),
        ("top1", ["top1", "top1_acc", "test_top1"], "Optional top-1 classification accuracy"),
        ("top5", ["top5", "top5_acc", "test_top5"], "Optional top-5 classification accuracy"),
    ],
    "uci": [
        ("accuracy", ["accuracy", "test_accuracy", "val_accuracy"], "Human activity recognition accuracy"),
        ("macro_f1", ["macro_f1", "test_macro_f1", "f1_macro"], "Macro-averaged F1 score"),
        ("per_class_accuracy", ["per_class_accuracy", "class_accuracy", "per_class_acc"], "Per-class accuracy breakdown if available"),
    ],
}

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
TITLE_FILL = PatternFill("solid", fgColor="D9EAF7")
ALT_FILL = PatternFill("solid", fgColor="F7FBFF")
WHITE_FONT = Font(name="Arial", color="FFFFFF", bold=True)
TITLE_FONT = Font(name="Arial", size=14, bold=True, color="1F4E78")
BASE_FONT = Font(name="Arial", size=10)
THIN_SIDE = Side(style="thin", color="B7B7B7")
THIN_BORDER = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
STATUS_FILLS = {
    "GOOD": PatternFill("solid", fgColor="C6EFCE"),
    "PRESENT": PatternFill("solid", fgColor="C6EFCE"),
    "REVIEW": PatternFill("solid", fgColor="FCE4D6"),
    "LOW": PatternFill("solid", fgColor="FFC7CE"),
    "NOT_AVAILABLE": PatternFill("solid", fgColor="E7E6E6"),
    "WARNING": PatternFill("solid", fgColor="FCE4D6"),
}

def normalize_key(value):
    return "".join(ch.lower() for ch in str(value) if ch.isalnum())

def load_json_file(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as exc:
        return None, f"WARNING: failed to read {path.name}: {exc}"

def ordered_json_files():
    priority = [
        "final_eval_summary.json",
        "eval_results.json",
        "retrieval_recall.json",
        "train_history.json",
        "ablation_summary.json",
    ]
    files = []
    seen = set()
    for name in priority:
        p = EXCEL_OUTPUT_DIR / name
        if p.exists() and p.is_file():
            files.append(p)
            seen.add(p.resolve())
    for p in sorted(EXCEL_OUTPUT_DIR.rglob("*.json")):
        try:
            key = p.resolve()
        except Exception:
            key = p
        if key not in seen:
            files.append(p)
            seen.add(key)
    return files

JSON_OBJECTS = []
JSON_WARNINGS = []
for json_path in ordered_json_files():
    obj, warning = load_json_file(json_path)
    if warning:
        JSON_WARNINGS.append(warning)
    else:
        JSON_OBJECTS.append((json_path.name, obj))

TXT_WARNINGS = []
for txt_path in sorted(list(EXCEL_OUTPUT_DIR.rglob("*.txt")) + list(EXCEL_OUTPUT_DIR.rglob("*.md"))):
    try:
        text = txt_path.read_text(encoding="utf-8", errors="replace")
        if "WARNING" in text.upper() or "NOT_AVAILABLE" in text.upper() or "NOT_RUN" in text.upper():
            TXT_WARNINGS.append(f"See {txt_path.name} for warnings/status notes")
    except Exception as exc:
        TXT_WARNINGS.append(f"WARNING: failed to read {txt_path.name}: {exc}")

def flatten_json(obj, prefix="", source=""):
    rows = []
    if prefix:
        rows.append((prefix, obj, source))
    if isinstance(obj, dict):
        for key, value in obj.items():
            child = f"{prefix}.{key}" if prefix else str(key)
            rows.extend(flatten_json(value, child, source))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            child = f"{prefix}.{idx}" if prefix else str(idx)
            rows.extend(flatten_json(value, child, source))
    return rows

FLAT_JSON = []
for source, obj in JSON_OBJECTS:
    FLAT_JSON.extend(flatten_json(obj, "", source))

def find_value(candidates):
    normalized_candidates = [normalize_key(c) for c in candidates]
    for cand in normalized_candidates:
        for key, value, source in FLAT_JSON:
            last_part = key.split(".")[-1]
            if normalize_key(last_part) == cand:
                return value, f"source: {source} / {key}"
    for cand in normalized_candidates:
        if len(cand) < 6:
            continue
        for key, value, source in FLAT_JSON:
            if normalize_key(key).endswith(cand):
                return value, f"source: {source} / {key}"
    return None, "NOT_AVAILABLE"

def safe_float(value):
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        try:
            return float(value)
        except Exception:
            return None
    return None

def display_value(value):
    if value is None:
        return "NOT_AVAILABLE"
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, (int, float)):
        return round(float(value), 6)
    if isinstance(value, (dict, list)):
        rendered = json.dumps(value, ensure_ascii=False)
        return rendered[:300] + ("..." if len(rendered) > 300 else "")
    return str(value)

def status_for(metric, value):
    if value is None or value == "NOT_AVAILABLE":
        return "NOT_AVAILABLE"
    if isinstance(value, (dict, list)):
        return "PRESENT" if value else "NOT_AVAILABLE"
    score = safe_float(value)
    if score is None:
        return "PRESENT"
    if score > 1.0 and score <= 100.0:
        score = score / 100.0
    metric_l = str(metric).lower()
    if "loss" in metric_l:
        if score <= 0.5:
            return "GOOD"
        if score <= 1.5:
            return "REVIEW"
        return "LOW"
    if score >= 0.70:
        return "GOOD"
    if score >= 0.40:
        return "REVIEW"
    return "LOW"

def get_by_alias(row, aliases):
    if not isinstance(row, dict):
        return None
    lookup = {normalize_key(k): v for k, v in row.items()}
    for alias in aliases:
        key = normalize_key(alias)
        if key in lookup:
            return lookup[key]
    return None

def find_train_history():
    candidates = [EXCEL_OUTPUT_DIR / "train_history.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*train*history*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        if isinstance(obj, list):
            return obj, path.name
        if isinstance(obj, dict):
            for key in ["history", "train_history", "rows"]:
                if isinstance(obj.get(key), list):
                    return obj[key], path.name
    return None, None

def build_training_rows():
    history, source = find_train_history()
    if not history:
        return ["Status", "Notes"], [["NOT_AVAILABLE", "train_history.json not found in output folder"]]
    alias_map = {
        "epoch": ["epoch"],
        "train_loss": ["train_loss", "loss_train", "training_loss"],
        "val_loss": ["val_loss", "test_loss", "valid_loss", "validation_loss"],
        "train_top1": ["train_top1", "train_accuracy", "train_acc", "train_i2t_acc", "train_top1_acc"],
        "val_top1": ["val_top1", "val_accuracy", "val_acc", "test_accuracy", "val_i2t_acc", "test_top1"],
        "lr": ["lr", "learning_rate"],
    }
    normalized_rows = []
    for raw in history:
        if not isinstance(raw, dict):
            continue
        row = {col: get_by_alias(raw, aliases) for col, aliases in alias_map.items()}
        normalized_rows.append(row)
    columns = [col for col in ["epoch", "train_loss", "val_loss", "train_top1", "val_top1", "lr"] if any(row.get(col) is not None for row in normalized_rows)]
    if "epoch" not in columns:
        columns = ["epoch"] + columns
    rows = [[display_value(row.get(col)) for col in columns] for row in normalized_rows]
    best_row = None
    best_note = None
    val_top1_values = [(safe_float(row.get("val_top1")), idx) for idx, row in enumerate(normalized_rows)]
    val_top1_values = [(v, idx) for v, idx in val_top1_values if v is not None]
    if val_top1_values:
        _, idx = max(val_top1_values, key=lambda x: x[0])
        best_row = normalized_rows[idx]
        best_note = "Best Epoch by val_top1"
    else:
        val_loss_values = [(safe_float(row.get("val_loss")), idx) for idx, row in enumerate(normalized_rows)]
        val_loss_values = [(v, idx) for v, idx in val_loss_values if v is not None]
        if val_loss_values:
            _, idx = min(val_loss_values, key=lambda x: x[0])
            best_row = normalized_rows[idx]
            best_note = "Best Epoch by val_loss"
    if best_row:
        best = [display_value(best_row.get(col)) for col in columns]
        if best:
            best[0] = f"Best Epoch: {best[0]}"
        rows.append(best)
        rows.append([best_note] + [""] * (len(columns) - 1))
    rows.append([f"source: {source}"] + [""] * (len(columns) - 1))
    return columns, rows

def find_ablation_object():
    candidates = [EXCEL_OUTPUT_DIR / "ablation_summary.json"]
    candidates.extend(sorted(EXCEL_OUTPUT_DIR.rglob("*ablation*.json")))
    seen = set()
    for path in candidates:
        try:
            key = path.resolve()
        except Exception:
            key = path
        if key in seen or not path.exists():
            continue
        seen.add(key)
        obj, warning = load_json_file(path)
        if warning:
            JSON_WARNINGS.append(warning)
            continue
        return obj, path.name
    return None, None

def setting_name(record, index):
    if not isinstance(record, dict):
        return f"setting_{index}"
    for key in ["setting", "name", "temperature", "lr", "learning_rate", "hidden_dim", "batch_size"]:
        if key in record:
            return f"{key}={record[key]}" if key not in ["setting", "name"] else str(record[key])
    return f"setting_{index}"

def build_ablation_rows():
    obj, source = find_ablation_object()
    if obj is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "No ablation JSON found in output folder"]]
    records = obj if isinstance(obj, list) else obj.get("results", obj.get("ablation", [obj])) if isinstance(obj, dict) else []
    if isinstance(records, dict):
        records = [records]
    rows = []
    for idx, record in enumerate(records):
        if not isinstance(record, dict):
            rows.append([f"setting_{idx}", "value", display_value(record), f"source: {source}"])
            continue
        flat = flatten_json(record, "", source)
        numeric_items = []
        for key, value, _ in flat:
            if safe_float(value) is not None and key.split(".")[-1].lower() not in ["epoch", "epochs", "seed"]:
                numeric_items.append((key, value))
        if numeric_items:
            for key, value in numeric_items[:12]:
                rows.append([setting_name(record, idx), key, display_value(value), f"source: {source}"])
        else:
            note = record.get("reason", record.get("note", "No numeric metric found"))
            status = record.get("status", "NOT_AVAILABLE")
            rows.append([setting_name(record, idx), "status", status, f"{note}; source: {source}"])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", f"No ablation records in {source}"]]

def find_per_class():
    for key, value, source in FLAT_JSON:
        if normalize_key(key.split(".")[-1]) in ["perclassaccuracy", "classaccuracy", "perclassacc"]:
            return value, source, key
    return None, None, None

def build_per_class_rows():
    value, source, key = find_per_class()
    if value is None:
        return [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE"]]
    rows = []
    if isinstance(value, dict):
        for cls, acc in value.items():
            rows.append([str(cls), display_value(acc), status_for("accuracy", acc)])
    elif isinstance(value, list):
        for idx, acc in enumerate(value):
            rows.append([str(idx), display_value(acc), status_for("accuracy", acc)])
    else:
        rows.append(["per_class_accuracy", display_value(value), status_for("accuracy", value)])
    rows.append([f"source: {source}", key, ""])
    return rows

def purpose_for(path):
    suffix = path.suffix.lower()
    if suffix == ".json":
        return "metrics/data"
    if suffix == ".png":
        return "figure/chart"
    if suffix in [".txt", ".md"]:
        return "report"
    if suffix == ".xlsx":
        return "excel report"
    if suffix == ".zip":
        return "compressed output"
    if suffix in [".pth", ".pt", ".ckpt"]:
        return "checkpoint"
    return "output artifact"

def build_manifest_rows():
    rows = []
    for path in sorted(EXCEL_OUTPUT_DIR.rglob("*")):
        if not path.is_file():
            continue
        try:
            rel = str(path.relative_to(EXCEL_OUTPUT_DIR))
        except Exception:
            rel = path.name
        try:
            size_kb = round(path.stat().st_size / 1024, 2)
        except Exception:
            size_kb = "NOT_AVAILABLE"
        rows.append([rel, path.suffix.lower() or "NO_EXTENSION", size_kb, purpose_for(path)])
    return rows or [["NOT_AVAILABLE", "NOT_AVAILABLE", "NOT_AVAILABLE", "Output folder is empty"]]

def autosize(ws):
    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_len = 0
        for cell in column_cells:
            try:
                max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 55)

def create_table_sheet(wb, title, headers, rows):
    ws = wb.create_sheet(title)
    ws.append([title] + [""] * (len(headers) - 1))
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(headers))
    title_cell = ws.cell(row=1, column=1)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    title_cell.alignment = Alignment(horizontal="center")
    ws.append([""] * len(headers))
    ws.append(headers)
    for row in rows:
        fixed = list(row)[:len(headers)] + [""] * max(0, len(headers) - len(row))
        ws.append(fixed)
    for row in ws.iter_rows():
        for cell in row:
            cell.font = BASE_FONT
            cell.border = THIN_BORDER
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    title_cell.font = TITLE_FONT
    title_cell.fill = TITLE_FILL
    for cell in ws[3]:
        cell.fill = HEADER_FILL
        cell.font = WHITE_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    status_cols = [idx + 1 for idx, name in enumerate(headers) if "status" in name.lower()]
    for row_idx in range(4, ws.max_row + 1):
        if row_idx % 2 == 0:
            for col_idx in range(1, len(headers) + 1):
                ws.cell(row=row_idx, column=col_idx).fill = ALT_FILL
        for col_idx in status_cols:
            value = str(ws.cell(row=row_idx, column=col_idx).value or "")
            fill = STATUS_FILLS.get(value)
            if fill:
                ws.cell(row=row_idx, column=col_idx).fill = fill
    ws.freeze_panes = "A4"
    autosize(ws)
    return ws

metric_rows = []
for metric, candidates, meaning in METRIC_SPECS[METRIC_SET]:
    value, note = find_value([metric] + candidates)
    metric_rows.append([metric, display_value(value), meaning, status_for(metric, value), note])
if JSON_WARNINGS or TXT_WARNINGS:
    for warning in JSON_WARNINGS[:5] + TXT_WARNINGS[:5]:
        metric_rows.append(["warning", warning, "Runtime/file warning", "WARNING", "Read from JSON/TXT files only"])

training_headers, training_rows = build_training_rows()
ablation_rows = build_ablation_rows()
per_class_rows = build_per_class_rows()

wb = Workbook()
wb.remove(wb.active)
create_table_sheet(wb, "Evaluation Metrics", ["Metric", "Value", "Benchmark/Meaning", "Status", "Notes"], metric_rows)
create_table_sheet(wb, "Training Summary", training_headers, training_rows)
create_table_sheet(wb, "Ablation Summary", ["Setting", "Metric", "Value", "Notes"], ablation_rows)
per_class_ws = create_table_sheet(wb, "Per-Class Breakdown", ["Class", "Accuracy", "Status"], per_class_rows)
per_class_ws.cell(row=1, column=1).value = "Per-Class / Breakdown"

# Save once so the manifest can include the Excel report itself.
wb.save(EXCEL_REPORT_PATH)
manifest_rows = build_manifest_rows()
create_table_sheet(wb, "Output Manifest", ["File name", "Extension", "Size KB", "Purpose"], manifest_rows)
wb.save(EXCEL_REPORT_PATH)

print(f"Excel report saved to: {EXCEL_REPORT_PATH}")
print(f"Total sheets: {len(wb.sheetnames)}")
print(f"Total output files listed: {len(manifest_rows)}")

## ADDON: Zip this notebook output

This cell zips this notebook's standardized output folder for direct Kaggle download. It skips missing folders safely and excludes large checkpoints over 100MB.

In [ ]:
import os
import zipfile
from pathlib import Path

def zip_folder(output_dir, zip_path, max_checkpoint_mb=100):
    output_dir = Path(output_dir)
    zip_path = Path(zip_path)

    if not output_dir.exists():
        print(f"WARNING: output folder does not exist: {output_dir}")
        return None

    added_files = []
    skipped_files = []

    def should_skip(file_path):
        suffix = file_path.suffix.lower()
        if suffix in [".pth", ".pt", ".ckpt"]:
            size_mb = file_path.stat().st_size / (1024 * 1024)
            if size_mb > max_checkpoint_mb:
                return True
        return False

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
        for file_path in output_dir.rglob("*"):
            if not file_path.is_file():
                continue

            if should_skip(file_path):
                skipped_files.append(str(file_path))
                continue

            arcname = file_path.relative_to(output_dir.parent)
            zipf.write(file_path, arcname)
            added_files.append(str(arcname))

    size_mb = zip_path.stat().st_size / (1024 * 1024)

    print("=" * 80)
    print(f"ZIP CREATED: {zip_path}")
    print(f"ZIP SIZE: {size_mb:.2f} MB")
    print(f"TOTAL FILES ADDED: {len(added_files)}")
    print("=" * 80)

    for f in added_files:
        print(f"- {f}")

    if skipped_files:
        print("=" * 80)
        print("SKIPPED LARGE CHECKPOINTS:")
        for f in skipped_files:
            print(f"- {f}")

    print("=" * 80)
    print("Download path:")
    print(str(zip_path))

    return zip_path

zip_folder(
    "/kaggle/working/ResultForSound1",
    "/kaggle/working/ResultForSound1.zip"
)